In [1]:
import pandas as pd
import numpy as np
import os
import zipfile
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

In [2]:
# Download and load the data
import keras
import os

f_path_1 = "train.csv"
url_1 = "https://jrssbcrsefilesnait.blob.core.windows.net/3950data1/train.csv.zip"
if not os.path.exists(f_path_1):
    file_1 = keras.utils.get_file(f_path_1, url_1)
train_df = pd.read_csv(f_path_1)

f_path_2 = "test.csv"
url_2 = "https://jrssbcrsefilesnait.blob.core.windows.net/3950data1/test.csv.zip"
if not os.path.exists(f_path_2):
    file_2 = keras.utils.get_file(f_path_2, url_2)
test_df = pd.read_csv(f_path_2)

## Project 1 - NLP and Text Classification

For this project you will need to classify some angry comments into their respective category of angry. The process that you'll need to follow is (roughly):
<ol>
<li> Use NLP techniques to process the training data. 
<li> Train model(s) to predict which class(es) each comment is in.
    <ul>
    <li> A comment can belong to any number of classes, including none. 
    </ul>
<li> Generate predictions for each of the comments in the test data. 
<li> Write your test data predicitions to a CSV file, which will be scored. 
</ol>

You can use any models and NLP libraries you'd like. 

## Training Data

Use the training data to train your prediction model(s). Each of the classification output columns (toxic to the end) is a human label for the comment_text, assessing if it falls into that category of "rude". A comment may fall into any number of categories, or none at all. Membership in one output category is <b>independent</b> of membership in any of the other classes (think about this when you plan on how to make these predictions - it may also make it easier to split work amongst a team...). 

In [3]:
#train_df = pd.read_csv("train.csv.zip")
train_df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\r\nWhy the edits made under my use...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\r\nMore\r\nI can't make any real suggestions...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


## Test Data

In [4]:
#test_df = pd.read_csv("test.csv")
test_df.head()

,id,comment_text
0,1,Yo bitch Ja Rule is more succesful then you'll...
1,2,== From RfC == \n\n The title is fine as it is...
2,3,""" \n\n == Sources == \n\n * Zawe Ashton on Lap..."
3,4,":If you have a look back at the source, the in..."
4,5,I don't anonymously edit articles at all.


In [ ]:
# Set Up 

# Column with the comment text
TEXT_COL = "comment_text"

# The 6 labels we must predict
LABEL_COLS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

# fill missing text values 
train_df[TEXT_COL] = train_df[TEXT_COL].fillna("")
test_df[TEXT_COL]  = test_df[TEXT_COL].fillna("")

# Features X and labels y
X_train = train_df[TEXT_COL]
y_train = train_df[LABEL_COLS].astype(int)

X_test = test_df[TEXT_COL]

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (159571,)
y_train shape: (159571, 6)
X_test shape: (153164,)


In [ ]:
# Building and Training

# A simple and strong baseline for text classification:
# TF-IDF converts text to numbers
# OneVsRest trains 1 classifier per label (multi-label classification)
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        max_features=100000,   
        ngram_range=(1, 2)     # unigrams + bigrams
    )),
    ("clf", OneVsRestClassifier(
        LogisticRegression(max_iter=2000, n_jobs=-1)
    ))
])

# Train the model
model.fit(X_train, y_train)

print("Training complete! YAAAAY")

Training complete! YAAAAY


In [11]:
# Train/validation split + model evaluation
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, hamming_loss, accuracy_score, classification_report

# Split only the training data into train/validation
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42
)

print("X_tr shape:", X_tr.shape)
print("X_val shape:", X_val.shape)
print("y_tr shape:", y_tr.shape)
print("y_val shape:", y_val.shape)

X_tr shape: (127656,)
X_val shape: (31915,)
y_tr shape: (127656, 6)
y_val shape: (31915, 6)


In [ ]:
# Validation model - same architecture as before, but will only be trained on the training split (X_tr, y_tr)
val_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        max_features=100000,
        ngram_range=(1, 2)
    )),
    ("clf", OneVsRestClassifier(
        LogisticRegression(max_iter=2000, n_jobs=-1)
    ))
])

# Train the validation model on the training split
val_model.fit(X_tr, y_tr)

print("Validation model trained!")

Validation model trained!


In [ ]:
# Evaluate with threshold = 0.5 first
# Predict probabilities on validation set
val_probs = val_model.predict_proba(X_val)

# Default threshold
THRESHOLD = 0.5
val_preds = (val_probs >= THRESHOLD).astype(int)

# Metrics
subset_accuracy = accuracy_score(y_val, val_preds)
f1_micro = f1_score(y_val, val_preds, average="micro", zero_division=0)
f1_macro = f1_score(y_val, val_preds, average="macro", zero_division=0)
ham_loss = hamming_loss(y_val, val_preds)

print("Metrics at threshold =", THRESHOLD)
print("Subset accuracy:", subset_accuracy)
print("F1 micro:", f1_micro)
print("F1 macro:", f1_macro)
print("Hamming loss:", ham_loss)

Metrics at threshold = 0.5
Subset accuracy: 0.9167162776124079
F1 micro: 0.6519594655187876
F1 macro: 0.46303113916502775
Hamming loss: 0.02026737688652149


In [17]:
# Let's see which labels are weak

# Detailed classification report for each label
for i, col in enumerate(LABEL_COLS):
    print("\n-----------------------")
    print("Label:", col)
    print(classification_report(
        y_val.iloc[:, i],
        val_preds[:, i],
        zero_division=0
    ))


-----------------------
Label: toxic
              precision    recall  f1-score   support

           0       0.96      0.99      0.98     28859
           1       0.92      0.58      0.71      3056

    accuracy                           0.95     31915
   macro avg       0.94      0.79      0.84     31915
weighted avg       0.95      0.95      0.95     31915


-----------------------
Label: severe_toxic
              precision    recall  f1-score   support

           0       0.99      1.00      1.00     31594
           1       0.59      0.18      0.28       321

    accuracy                           0.99     31915
   macro avg       0.79      0.59      0.64     31915
weighted avg       0.99      0.99      0.99     31915


-----------------------
Label: obscene
              precision    recall  f1-score   support

           0       0.98      1.00      0.99     30200
           1       0.93      0.59      0.72      1715

    accuracy                           0.98     31915
   ma

In [18]:
# Testing tresholds

thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]
results = []

for th in thresholds:
    preds_th = (val_probs >= th).astype(int)

    subset_acc = accuracy_score(y_val, preds_th)
    f1_micro = f1_score(y_val, preds_th, average="micro", zero_division=0)
    f1_macro = f1_score(y_val, preds_th, average="macro", zero_division=0)
    ham_loss = hamming_loss(y_val, preds_th)

    results.append([th, subset_acc, f1_micro, f1_macro, ham_loss])

results_df = pd.DataFrame(
    results,
    columns=["threshold", "subset_accuracy", "f1_micro", "f1_macro", "hamming_loss"]
)

results_df.sort_values(by="f1_micro", ascending=False)

,threshold,subset_accuracy,f1_micro,f1_macro,hamming_loss
1,0.2,0.909510,0.729116,0.570021,0.019677
2,0.3,0.918032,0.718641,0.538249,0.018507
3,0.4,0.918722,0.690456,0.506797,0.019071
0,0.1,0.864766,0.679302,0.560473,0.028983
4,0.5,0.916716,0.651959,0.463031,0.020267
5,0.6,0.914241,0.607991,0.409938,0.021724


The best global threshold is 0.2, because it gives the highest F1_micro = 0.729116, which is usually the most useful metric here for multi-label classification.

In [ ]:
#Predict and build output

# Predict probabilities for each label

probs = model.predict_proba(X_test)

# Convert probabilities -to 0/1 using threshold
THRESHOLD = 0.2
preds = (probs >= THRESHOLD).astype(int)

# Build output dataframe - 6 cols
out_df = pd.DataFrame(preds, columns=LABEL_COLS)

# Add id column
out_df.insert(0, "id", test_df["id"].values)


out_df.head()

,id,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,1,1,0,1,0,1,0
1,2,0,0,0,0,0,0
2,3,0,0,0,0,0,0
3,4,0,0,0,0,0,0
4,5,0,0,0,0,0,0


In [22]:
out_df[LABEL_COLS].sum()

toxic            36521
severe_toxic      2494
obscene          20700
threat             468
insult           19152
identity_hate     2223
dtype: int64

In [ ]:
# Comparison of label counts in training vs predicted test set
print("Training label counts:")
print(train_df[LABEL_COLS].sum())

print("\nPredicted test label counts:")
print(out_df[LABEL_COLS].sum())

Training label counts:
toxic            15294
severe_toxic      1595
obscene           8449
threat             478
insult            7877
identity_hate     1405
dtype: int64

Predicted test label counts:
toxic            36521
severe_toxic      2494
obscene          20700
threat             468
insult           19152
identity_hate     2223
dtype: int64


In [ ]:
# save output

# Make output folder
os.makedirs("output", exist_ok=True)

# Save the CSV 
csv_path = "output/out.csv"
out_df.to_csv(csv_path, index=False)
print("Saved:", csv_path)

# make a zip too
zip_path = "out.csv.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(csv_path, arcname="out.csv")

print("Saved:", zip_path)

Saved: output/out.csv
Saved: out.csv.zip


## Output Details, Submission Info, and Example Submission

For this project, please output your predictions in a CSV file. The structure of the CSV file should match the structure of the example below. 

The output should contain one row for each row of test data, complete with the columns for ID and each classification.

Into Moodle please submit:
<ul>
<li> Your notebook file(s). I'm not going to run them, just look. 
<li> Your sample submission CSV. This will be evaluated for accuracy against the real labels; only a subset of the predictions will be scored. 
</ul>

It is REALLY, REALLY, REALLY important the the structure of your output matches the specifications. The accuracies will be calculated by a script, and it is expecting a specific format. 

### Sample Evaluator

The file prediction_evaluator.ipynb contains an example scoring function, scoreChecker. This function takes a sumbission and an answer key, loops through, and evaluates the accuracy. You can use this to verify the format of your submission. I'm going to use the same function to evaluate the accuracy of your submission, against the answer key (unless I made some mistake in this counting function).

In [9]:
#Construct dummy data for a sample output. 
#You won't do this part, you have real data
#Your data should have the same structure, so the CSV output is the same
dummy_ids = ["dfasdf234", "asdfgw43r52", "asdgtawe4", "wqtr215432"]
dummy_toxic = [0,0,0,0]
dummy_severe = [0,0,0,0]
dummy_obscene = [0,1,1,0]
dummy_threat = [0,1,0,1]
dummy_insult = [0,0,1,0]
dummy_ident = [0,1,1,0]
columns = ["id", "toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
sample_out = pd.DataFrame( list(zip(dummy_ids, dummy_toxic, dummy_severe, dummy_obscene, dummy_threat, dummy_insult, dummy_ident)),
                    columns=columns)
sample_out.head()

,id,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,dfasdf234,0,0,0,0,0,0
1,asdfgw43r52,0,0,1,1,0,1
2,asdgtawe4,0,0,1,0,1,1
3,wqtr215432,0,0,0,1,0,0


In [10]:
#Write DF to CSV. Please keep the "out.csv" filename. Moodle will auto-preface it with an identifier when I download it. 
#This command should work with your dataframe of predictions. 
sample_out.to_csv('output/out.csv', index=False)  

## Grading

The grading for this is split between accuracy and well written code:
<ul>
<li> 75% - Accuracy. The most accurate will get 100% on this, the others will be scaled down from there. 
<li> 25% - Code quality. Can the code be followed and made sense of - i.e. comments, sections, titles. 
</ul>